# HW3

In [1]:
import pandas as pd
import altair as alt

df_2324 = pd.read_csv("PL-season-2324.csv")
df_2425 = pd.read_csv("PL-season-2425.csv")
df_2324["Season"] = "2023-24"
df_2425["Season"] = "2024-25"
df = pd.concat([df_2324, df_2425], ignore_index=True)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)
df.head()

/var/folders/pt/gd5dg1v51y5fdd2jfpf9wj040000gn/T/ipykernel_18824/3971898688.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], dayfirst=True)


,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,...,AST,HF,AF,HC,AC,HY,AY,HR,AR,Season
0,2023-08-11,Burnley,Man City,0,3,A,0,2,A,C Pawson,...,8,11,8,6,5,0,0,1,0,2023-24
1,2023-08-12,Arsenal,Nott'm Forest,2,1,H,2,0,H,M Oliver,...,2,12,12,8,3,2,2,0,0,2023-24
2,2023-08-12,Bournemouth,West Ham,1,1,D,0,0,D,P Bankes,...,3,9,14,10,4,1,4,0,0,2023-24
3,2023-08-12,Brighton,Luton,4,1,H,1,0,H,D Coote,...,3,11,12,6,7,2,2,0,0,2023-24
4,2023-08-12,Everton,Fulham,0,1,A,0,0,D,S Attwell,...,2,12,6,10,4,0,2,0,0,2023-24


## Data Preparation

Three datasets are derived from the raw match-level data:

- **`team_matches`** One row per *(team, game)* across both seasons, with attacking stats and a per-team game number 1–38. Used for Q2.
- **`team_ha`** Total home and away points per *(team, season)*. Used for Q3.
- **`matches`** Match-level data with total goals and a human-readable label. Used for Q4.

In [ ]:
home = df.assign(
    Team=df["HomeTeam"],
    Goals=df["FTHG"],
    Points=df["FTR"].map({"H": 3, "D": 1, "A": 0}),
    Shots=df["HS"],
    ShotsOT=df["HST"],
    Corners=df["HC"]
)[ ["Date", "Season", "Team", "Goals", "Points", "Shots", "ShotsOT", "Corners"] ]

away = df.assign(
    Team=df["AwayTeam"],
    Goals=df["FTAG"],
    Points=df["FTR"].map({"A": 3, "D": 1, "H": 0}),
    Shots=df["AS"],
    ShotsOT=df["AST"],
    Corners=df["AC"]
)[ ["Date", "Season", "Team", "Goals", "Points", "Shots", "ShotsOT", "Corners"] ]

team_matches = (
    pd.concat([home, away])
    .sort_values(["Season", "Team", "Date"])
    .reset_index(drop=True)
)
team_matches["GameNum"] = team_matches.groupby(["Season", "Team"]).cumcount() + 1

home_pts = (
    df.assign(Team=df["HomeTeam"], Points=df["FTR"].map({"H": 3, "D": 1, "A": 0}))
    .groupby(["Season", "Team"])["Points"].sum().rename("HomePoints").reset_index()
)
away_pts = (
    df.assign(Team=df["AwayTeam"], Points=df["FTR"].map({"A": 3, "D": 1, "H": 0}))
    .groupby(["Season", "Team"])["Points"].sum().rename("AwayPoints").reset_index()
)
team_ha = home_pts.merge(away_pts, on=["Season", "Team"])

matches = df[["Date", "Season", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]].copy()
matches["TotalGoals"] = matches["FTHG"] + matches["FTAG"]
matches["MatchLabel"] = (
    matches["HomeTeam"] + " " +
    matches["FTHG"].astype(str) + "-" +
    matches["FTAG"].astype(str) + " " +
    matches["AwayTeam"]
)

team_ha.head()

team_matches: (1520, 9)  team_ha: (40, 4)  matches: (760, 9)


,Season,Team,HomePoints,AwayPoints
0,2023-24,Arsenal,47,42
1,2023-24,Aston Villa,40,28
2,2023-24,Bournemouth,27,21
3,2023-24,Brentford,22,17
4,2023-24,Brighton,30,18


## Dashboard

The full dashboard is rendered in a single code cell so all three charts share the same Altair selection objects.

The dashed diagonal in Q3 is the *y = x* line; dots above it earned more points away than at home.

In [3]:
team_sel = alt.selection_point(fields=["Team"], empty=False)
brush = alt.selection_interval()

metric_param = alt.param(
    name="metric",
    value="Goals",
    bind=alt.binding_select(
        options=["Goals", "Shots", "ShotsOT", "Corners"],
        labels=["Goals", "Shots", "Shots on Target", "Corners"],
        name="Metric: "
    )
)

diag = (
    alt.Chart(pd.DataFrame({"x": [0, 45], "y": [0, 45]}))
    .mark_line(strokeDash=[5, 3], color="gray", opacity=0.35)
    .encode(x="x:Q", y="y:Q")
)

q3_dots = (
    alt.Chart(team_ha)
    .mark_circle(size=90)
    .encode(
        x=alt.X("HomePoints:Q", title="Home Points"),
        y=alt.Y("AwayPoints:Q", title="Away Points"),
        color=alt.condition(team_sel, alt.Color("Season:N"), alt.value("lightgray")),
        opacity=alt.condition(team_sel, alt.value(0.9), alt.value(0.45)),
        tooltip=["Team:N", "Season:N", "HomePoints:Q", "AwayPoints:Q"]
    )
    .add_params(team_sel)
)

q3 = (
    alt.layer(diag, q3_dots)
    .properties(
        title="Q3: Home vs. Away Points — click a team to explore its attacking trends",
        width=360,
        height=300
    )
)

q2 = (
    alt.Chart(team_matches)
    .transform_filter(team_sel)
    .transform_fold(["Goals", "Shots", "ShotsOT", "Corners"], as_=["Metric", "Value"])
    .transform_filter("datum.Metric === metric")
    .transform_window(
        RollingAvg="mean(Value)",
        frame=[-4, 0],
        sort=[{"field": "GameNum"}],
        groupby=["Season", "Team", "Metric"]
    )
    .mark_line(point=True)
    .encode(
        x=alt.X("GameNum:Q", title="Game Number"),
        y=alt.Y("RollingAvg:Q", title="5-Game Rolling Average"),
        color=alt.Color("Season:N"),
        detail="Team:N",
        tooltip=[
            alt.Tooltip("Team:N"),
            alt.Tooltip("GameNum:Q", title="Game"),
            alt.Tooltip("Season:N"),
            alt.Tooltip("RollingAvg:Q", format=".2f", title="Rolling Avg")
        ]
    )
    .add_params(metric_param)
    .properties(
        title="Q2: Attacking Performance Over Time (select team above · change metric below)",
        width=410,
        height=300
    )
)

q4_scatter = (
    alt.Chart(matches)
    .mark_circle(size=55)
    .encode(
        x=alt.X("FTHG:Q", title="Home Goals", axis=alt.Axis(tickMinStep=1)),
        y=alt.Y("FTAG:Q", title="Away Goals", axis=alt.Axis(tickMinStep=1)),
        color=alt.condition(brush, alt.Color("Season:N"), alt.value("lightgray")),
        opacity=alt.condition(brush, alt.value(0.85), alt.value(0.25)),
        tooltip=["HomeTeam:N", "AwayTeam:N", "FTHG:Q", "FTAG:Q", "Season:N"]
    )
    .add_params(brush)
    .properties(
        title="Q4: Match Outcomes — drag to brush and explore extreme matches",
        width=330,
        height=300
    )
)

q4_detail = (
    alt.Chart(matches)
    .transform_filter(brush)
    .transform_window(
        rank="rank()",
        sort=[{"field": "TotalGoals", "order": "descending"}]
    )
    .transform_filter("datum.rank <= 15")
    .mark_bar()
    .encode(
        x=alt.X("TotalGoals:Q", title="Total Goals"),
        y=alt.Y("MatchLabel:N", sort="-x", title=""),
        color=alt.Color("Season:N"),
        tooltip=["HomeTeam:N", "AwayTeam:N", "FTHG:Q", "FTAG:Q", "TotalGoals:Q"]
    )
    .properties(
        title="Q4: Top 15 Brushed Matches by Total Goals",
        width=410,
        height=300
    )
)

dashboard = (
    alt.vconcat(
        alt.hconcat(q3, q2).resolve_scale(color="independent"),
        alt.hconcat(q4_scatter, q4_detail).resolve_scale(color="independent")
    )
    .configure_view(strokeWidth=0)
    .configure_title(fontSize=12, fontWeight="normal")
    .configure_axis(labelFontSize=11, titleFontSize=12)
    .configure_legend(labelFontSize=11, titleFontSize=12)
)

dashboard

alt.VConcatChart(...)

Interactivity is what makes this dashboard analytically useful rather than merely decorative: clicking a team in Q3 immediately filters Q2 to reveal the dynamics behind its season summary - whether it started strong and faded or built momentum — offloading the cognitive work of cross-referencing two charts, while the metric toggle lets users interrogate goals, shots, shots on target, and corners without needing four separate views. Without these links, Q2 would require 40 overlapping lines (unreadable), and the Q4 brush-to-detail connection would be impossible in static form since brushing collapses a geometric region of extreme matches into a ranked list that a fixed chart simply cannot replicate. Several decisions went beyond the technical specifications. The dashed y = x diagonal in Q3 makes home/away asymmetry legible at a glance (dots above it earned more away points than home points). A 5-game rolling window was chosen because 3 games is too reactive to individual results while 10 games erases meaningful mid-season shape, and the Q4 detail view is capped at 15 to stay readable when the brush covers a dense region. The trickiest technical challenge was ordering Altair transforms correctly — transform_fold must precede the param filter (datum.Metric === metric) so the Metric field exists when the signal comparison runs, and both must precede transform_window so the rolling mean operates on the already-filtered, already-folded data.